# 🧱 Lab: Fix the Broken Repo (4 Seeded Bugs)

**Module 5: Debugging Lab 1: Read the Failure** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Reproduce** every failure in an unfamiliar repo from a clean starting copy
2. **Localise** each bug to the file (pipeline stage) where it lives, using the debugging decision tree
3. **Fix** four seeded bugs, one change at a time, until `verify.py` is green from a fresh copy
4. **Write** a one-sentence cause and fix for each bug

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| The debugging decision tree | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. | mini-walk-debug-tree |
| Silent failure | An empty filter result or coerced dtype can silently change what your code is operating on, producing confident, wrong output without any exception or traceback. | mini-fix-data-failures |
| Reproducibility | Someone else's machine reproduces your result from documented, pinned steps, starting from a fresh repo clone with no hidden setup. | lab-clone-and-run-check |

## Prerequisites

- **mini-walk-debug-tree**: The printable decision tree
- **mini-fix-import-failures**: Import and environment failures
- **mini-fix-data-failures**: Silent data-shaped failures

## Scenario

`broken_repo/` is a small Python repo that processes sales data. It has **four seeded bugs**: some crash loudly, some fail silently. Your goal is to fix them one at a time until `verify.py` passes every check.

| File | Pipeline stage |
|------|----------------|
| `loader.py` | Load: returns the sales records |
| `analysis.py` | Analyse: total revenue and top product |
| `report.py` | Report: formats the final summary |
| `verify.py` | Not a stage: runs 5 checks and prints PASS/FAIL for each |

## Step 0 — Create the Broken Repo

This cell writes a clean `broken_repo/`, with the seeded bugs, so every run starts from a fresh copy.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import display, Markdown

# Start from a clean copy every time, like a fresh clone
repo_dir = "broken_repo"
shutil.rmtree(repo_dir, ignore_errors=True)
repo = Path(repo_dir)
repo.mkdir()

FILES = {
    "loader.py": '''"""Load sales data from a CSV-like source."""

from collections import OrderableDict


def load_sales():
    """Return a list of sales records (simulating a CSV read)."""
    raw_rows = [
        {"product": "Widget", "quantity": "15", "unit_price": 9.99},
        {"product": "Gadget", "quantity": "8",  "unit_price": 24.50},
        {"product": "Widget", "quantity": "22", "unit_price": 9.99},
        {"product": "Gizmo",  "quantity": "5",  "unit_price": 49.99},
        {"product": "Gadget", "quantity": "12", "unit_price": 24.50},
    ]
    return raw_rows
''',
    "analysis.py": '''"""Compute sales metrics."""


def total_revenue(sales):
    """Return the total revenue across all sales."""
    return sum(row["quantity"] * row["unit_price"] for row in sales)


def top_product(sales):
    """Return the product name with the highest total revenue."""
    revenue_by_product = {}
    for row in sales:
        name = row["product"]
        revenue_by_product[name] = revenue_by_product.get(name, 0) + (
            row["quantity"] * row["unit_price"]
        )
    return min(revenue_by_product, key=revenue_by_product.get)
''',
    "report.py": '''"""Format the final sales report."""


def build_report(total, top):
    """Return a formatted report string."""
    lines = [
        "=== Sales Report ===",
        f"Total Revenue: ${total:,.2f}",
        f"Top Product:   {top_product_name}",
        "===================",
    ]
    return "\\n".join(lines)
''',
}

# Write each file into the repo folder
for name, code in FILES.items():
    (repo / name).write_text(code, encoding="utf-8")

print(f"Repo written: {repo_dir}/ with {len(FILES)} files")

Repo written: broken_repo/ with 3 files


`verify.py` is the checker: run it after every change, and never edit it.

In [2]:
VERIFY = '''"""Run 5 checks and print PASS/FAIL for each."""
EXPECTED_REVENUE = (15 + 22) * 9.99 + (8 + 12) * 24.50 + 5 * 49.99
results = []


def check(name, test):
    try:
        ok, detail = test()
    except Exception as e:
        # keep the message, drop the file path Python appends to import errors
        ok, detail = False, f"{type(e).__name__}: {str(e).split(' (')[0]}"
    results.append(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + ("" if ok else f"  —  {detail}"))


def sales():
    from loader import load_sales
    return load_sales()


def revenue():
    from analysis import total_revenue
    return total_revenue(sales())


def top():
    from analysis import top_product
    return top_product(sales())


def report():
    from report import build_report
    return build_report(revenue(), top())


check("loader imports", lambda: (len(sales()) == 5, "expected 5 rows"))
check("quantities are int", lambda: (all(isinstance(r["quantity"], int) for r in sales()), f"got {type(sales()[0]['quantity']).__name__}"))
check("total revenue correct", lambda: (abs(revenue() - EXPECTED_REVENUE) < 0.01, f"expected {EXPECTED_REVENUE:.2f}, got {revenue():.2f}"))
check("top product correct", lambda: (top() == "Gadget", f"expected Gadget, got {top()}"))
check("report builds correctly", lambda: ("Gadget" in report() and f"{EXPECTED_REVENUE:,.2f}" in report(), "report is missing the total or the top product"))

print()
print(f"{sum(results)}/{len(results)} checks passed")
print("✅ All green — repo is fixed!" if all(results) else "❌ Keep going — some checks still fail.")
'''

(repo / "verify.py").write_text(VERIFY, encoding="utf-8")
print(f"Wrote {repo_dir}/verify.py")

Wrote broken_repo/verify.py


## Helpers and the Debug Log

Here are small helpers to run verify, view files, and apply one safe patch per fix:

- `run_verify()`: runs `verify.py` in a fresh Python process
- `show(file)`: prints a file with line numbers
- `patch(file, old, new)`: makes one change and fails if `old` is missing
- `md(text)`: renders a markdown table
- `debug_log`: stores a one-sentence cause and fix per bug

After the helpers, we run verify once as the baseline. That run is also the **Reproduce** step for Bug 1.

In [3]:
def run_verify(cwd=repo_dir):
    """Run verify.py inside a repo folder in a fresh Python process and print its output."""
    result = subprocess.run(
        [sys.executable, "verify.py"],
        cwd=cwd,
        capture_output=True,
        text=True,
        encoding="utf-8",   # read the ✅ / ❌ output as UTF-8
    )
    print(result.stdout)
    return result.returncode


def show(filename):
    """Print one repo file with line numbers."""
    lines = (repo / filename).read_text(encoding="utf-8").splitlines()
    for number, line in enumerate(lines, 1):
        print(f"{number:>3}  {line}")


def patch(filename, old, new):
    """Make ONE change to one repo file; fail loudly if the old text is missing."""
    path = repo / filename
    code = path.read_text(encoding="utf-8")
    assert old in code, f"{old!r} not found in {filename}"
    path.write_text(code.replace(old, new, 1), encoding="utf-8")


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# One entry per bug: the error, its cause, and the fix
debug_log = []

In [4]:
run_verify()

  [FAIL] loader imports  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] quantities are int  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] total revenue correct  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] top product correct  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] report builds correctly  —  ImportError: cannot import name 'OrderableDict' from 'collections'

0/5 checks passed
❌ Keep going — some checks still fail.



0

---

## Bug 1 — `loader.py` Won't Import (Loud Failure)

**Reproduce.** The baseline run shows every check failing with the same `ImportError`.

**Localise.** The first failing check is `loader imports`, and it points to `from collections import OrderableDict` in `loader.py`.

**Shrink.** Already minimal: one import line.

**Change one thing.** Fix the typo in that name, then re-run verify.

In [5]:
# Fix the typo in the import, then re-run verify
patch("loader.py", "OrderableDict", "OrderedDict")   # the ONE change
debug_log.append({
    "bug": "Bug 1",
    "error": "ImportError: cannot import name 'OrderableDict'",
    "cause": "The import has a typo: collections has no OrderableDict.",
    "fix": "Changed the import to OrderedDict.",
})
run_verify()

  [PASS] loader imports
  [FAIL] quantities are int  —  got str
  [FAIL] total revenue correct  —  TypeError: can't multiply sequence by non-int of type 'float'
  [FAIL] top product correct  —  TypeError: can't multiply sequence by non-int of type 'float'
  [FAIL] report builds correctly  —  TypeError: can't multiply sequence by non-int of type 'float'

1/5 checks passed
❌ Keep going — some checks still fail.



0

---

## Bug 2 — Quantities Are Strings (Silent Failure)

`quantities are int` fails with `got str`. It is a silent, data-shaped bug: nothing raised in the loader, and the wrong dtype travelled on until the arithmetic broke. The values are already wrong when they leave `loader.py`, so we localise there.

In [6]:
show("loader.py")

  1  """Load sales data from a CSV-like source."""
  2  
  3  from collections import OrderedDict
  4  
  5  
  6  def load_sales():
  7      """Return a list of sales records (simulating a CSV read)."""
  8      raw_rows = [
  9          {"product": "Widget", "quantity": "15", "unit_price": 9.99},
 10          {"product": "Gadget", "quantity": "8",  "unit_price": 24.50},
 11          {"product": "Widget", "quantity": "22", "unit_price": 9.99},
 12          {"product": "Gizmo",  "quantity": "5",  "unit_price": 49.99},
 13          {"product": "Gadget", "quantity": "12", "unit_price": 24.50},
 14      ]
 15      return raw_rows


`quantity` is text (e.g. `"15"`), so we convert it to `int` at the loader boundary.

In [7]:
patch(
    "loader.py",
    "    return raw_rows",
    '    return [{**row, "quantity": int(row["quantity"])} for row in raw_rows]',
)   # the ONE change
debug_log.append({
    "bug": "Bug 2",
    "error": "Silent: quantities are str, so later arithmetic breaks",
    "cause": "The loader returned the CSV text values without converting them.",
    "fix": "Converted quantity to int as each row leaves load_sales().",
})
run_verify()

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [FAIL] top product correct  —  expected Gadget, got Gizmo
  [FAIL] report builds correctly  —  NameError: name 'top_product_name' is not defined

3/5 checks passed
❌ Keep going — some checks still fail.


0

---

## Bug 3 — Wrong Top Product (Silent Failure)

The check expects `Gadget` but gets `Gizmo`. Revenue is correct, so the loader is fine, and we localise to `top_product` in `analysis.py`.

In [8]:
show("analysis.py")

  1  """Compute sales metrics."""
  2  
  3  
  4  def total_revenue(sales):
  5      """Return the total revenue across all sales."""
  6      return sum(row["quantity"] * row["unit_price"] for row in sales)
  7  
  8  
  9  def top_product(sales):
 10      """Return the product name with the highest total revenue."""
 11      revenue_by_product = {}
 12      for row in sales:
 13          name = row["product"]
 14          revenue_by_product[name] = revenue_by_product.get(name, 0) + (
 15              row["quantity"] * row["unit_price"]
 16          )
 17      return min(revenue_by_product, key=revenue_by_product.get)


`min` picks the lowest-revenue product, so we change it to `max`.

In [9]:
patch("analysis.py", "return min(", "return max(")   # the ONE change
debug_log.append({
    "bug": "Bug 3",
    "error": "Silent: top product is Gizmo instead of Gadget",
    "cause": "top_product used min(), which returns the lowest-revenue product.",
    "fix": "Changed min() to max().",
})
run_verify()

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [FAIL] report builds correctly  —  NameError: name 'top_product_name' is not defined

4/5 checks passed
❌ Keep going — some checks still fail.


0

---

## Bug 4 — Report Crashes with `NameError` (Loud Failure)

`report builds correctly` fails with `NameError: name 'top_product_name' is not defined`. That name appears only in the f-string in `report.py`, so the fix is to use the parameter `top` there.

In [10]:
patch("report.py", "{top_product_name}", "{top}")   # the ONE change
debug_log.append({
    "bug": "Bug 4",
    "error": "NameError: name 'top_product_name' is not defined",
    "cause": "The f-string used an old variable name instead of the parameter top.",
    "fix": "Changed {top_product_name} to {top}.",
})
run_verify()

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [PASS] report builds correctly

5/5 checks passed
✅ All green — repo is fixed!



0

## Final Verification — From a Fresh Copy

Copy the repo to a fresh folder and run verify again: all checks must pass.

In [11]:
clone_dir = "broken_repo_clone"
shutil.rmtree(clone_dir, ignore_errors=True)
shutil.copytree(repo_dir, clone_dir, ignore=shutil.ignore_patterns("__pycache__"))

return_code = run_verify(cwd=clone_dir)

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [PASS] report builds correctly

5/5 checks passed
✅ All green — repo is fixed!



## The Debug Log

Here is the debug log (cause + fix per bug), then we clean up.

In [12]:
rows = "| Bug | Error | Cause | Fix |\n|-----|-------|-------|-----|\n"
for entry in debug_log:
    rows += f"| {entry['bug']} | {entry['error']} | {entry['cause']} | {entry['fix']} |\n"
md(rows)

shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(clone_dir, ignore_errors=True)
print(f"Cleaned up {repo_dir}/ and {clone_dir}/")

| Bug | Error | Cause | Fix |
|-----|-------|-------|-----|
| Bug 1 | ImportError: cannot import name 'OrderableDict' | The import has a typo: collections has no OrderableDict. | Changed the import to OrderedDict. |
| Bug 2 | Silent: quantities are str, so later arithmetic breaks | The loader returned the CSV text values without converting them. | Converted quantity to int as each row leaves load_sales(). |
| Bug 3 | Silent: top product is Gizmo instead of Gadget | top_product used min(), which returns the lowest-revenue product. | Changed min() to max(). |
| Bug 4 | NameError: name 'top_product_name' is not defined | The f-string used an old variable name instead of the parameter top. | Changed {top_product_name} to {top}. |


Cleaned up broken_repo/ and broken_repo_clone/


## 🎯 Summary

### What You Built

You fixed four seeded bugs in an unfamiliar repo, two loud and two silent, until `verify.py` was green and stayed green from a fresh copy.

### Key Takeaways

1. **Start with the verification script**: it shows which checks fail and which bug to fix first
2. **Loud failures first**: an import error at the top blocks every check downstream
3. **Silent failures need expected values**: string quantities and `min` vs `max` only showed up against known-good answers
4. **One change at a time, re-verify**: each re-run confirms the fix and reveals the next bug
5. **Prove it from a fresh copy**: the repo is fixed only when a clean copy passes the same checks